# Cox-Ingersoll-Ross Model — Positivity, Simulation and Calibration
## QRE-45

The Cox-Ingersoll-Ross (1985) model replaces Vasicek's constant diffusion $\sigma\,dW$ with a **square-root diffusion** $\sigma\sqrt{r}\,dW$. This single change has three far-reaching consequences:

1. **Rates are guaranteed non-negative** — when the Feller condition holds, $r(t) > 0$ almost surely
2. **Volatility is state-dependent** — high rates are more volatile, near-zero rates are nearly deterministic
3. **The transition distribution is non-central chi-squared** — not Gaussian, so exact simulation is structurally different from Vasicek

**What this notebook adds beyond QRE-43 and QRE-44:**
<small>

| Topic | Vasicek (QRE-43) | HW (QRE-44) | New in QRE-45 |
|---|---|---|---|
| Non-negative rates | No | No | **Yes — Feller condition** |
| Non-Gaussian transition | No | No | **Yes — χ² exact simulation** |
| State-dependent vol | No | No | **Yes — $\sigma\sqrt{r}$ diffusion** |
| Affine ZCB formula | Yes ($B$, $\alpha$) | Yes (market-fitted) | **Yes — different $B$, $A$** |
| Calibration to OIS | Yes | Via $\theta(t)$ | **Yes — same least-squares approach** |
</small>

**Prerequisites:** [QRE-43 Vasicek](05_vasicek_model.ipynb) for affine term structures and calibration mechanics; [QRE-46 CIRProcess](../../src/quant_risk/models/rates.py) for the production implementation.


In [ ]:
from scipy.optimize import minimize
from scipy import stats
from scipy.stats import ncx2, gamma as gamma_dist

from quant_risk.setup import base
from quant_risk.config import PROCESSED_DIR
from quant_risk.curves.ois import OISCurve
from quant_risk.models.rates import CIRProcess

np, pd, plt = base()

RNG_SEED = 42

---
## 1. The CIR Model — Square-Root Diffusion

### 1.1 Stochastic Differential Equation

$$dr(t) = \kappa\bigl(\theta - r(t)\bigr)\,dt + \sigma\sqrt{r(t)}\,dW(t)$$
<small>

| Symbol | Meaning | Units | Notes |
|---|---|---|---|
| $\kappa > 0$ | Mean reversion speed | year$^{-1}$ | Same role as Vasicek |
| $\theta$ | Long-run mean rate | % | Same role as Vasicek |
| $\sigma > 0$ | Volatility coefficient | $\sqrt{\%}$/year$^{1/2}$ | **Different units from Vasicek's $\sigma$** |
</small>

The drift $\kappa(\theta-r)$ is **identical to Vasicek** — so the mean of $r(t)$ evolves exactly as in Vasicek. What changes is the **diffusion term**: $\sigma\sqrt{r}$ instead of $\sigma$.

### 1.2 Why $\sqrt{r}$ Prevents Negative Rates

When $r(t) \to 0$, the diffusion $\sigma\sqrt{r} \to 0$. The stochastic shocks vanish, leaving only the **deterministic drift** $\kappa(\theta - 0) = \kappa\theta > 0$ pulling the rate back up. The origin becomes a reflecting barrier.

More precisely, the **Feller condition**:

$$2\kappa\theta > \sigma^2$$

(in the consistent units: $2\kappa[\%\text{/year}]\cdot\theta[\%] > \sigma^2[\%/\text{year}]$)

When satisfied, the degrees of freedom $d = 4\kappa\theta/\sigma^2 > 2$ and $r(t) > 0$ almost surely for all $t$. When $0 < 2\kappa\theta \leq \sigma^2$, zero is accessible but the process remains non-negative.

### 1.3 State-Dependent Volatility — a CIR Insight

In Vasicek, $\text{Std}[dr] = \sigma\,\sqrt{dt}$ — constant regardless of the rate level. In CIR, $\text{Std}[dr] = \sigma\sqrt{r}\,\sqrt{dt}$ — proportional to $\sqrt{r}$.

This means:
- At $r = \theta = 2.5\%$: $\text{vol}(r) = \sigma\sqrt{2.5}$ per $\sqrt{\text{year}}$
- At $r = 5\%$ (rate doubling): $\text{vol}(r) = \sigma\sqrt{5} = \sqrt{2}$ times higher
- At $r \to 0$: vol collapses — the near-zero floor is deterministic

For a Vasicek model calibrated to match CIR's vol at $\theta$:
$$\sigma_{\text{Vasicek}} = \sigma_{\text{CIR}}\sqrt{\theta}$$

### 1.4 Distributional Properties

| Property | CIR | Vasicek |
|---|---|---|
| $\mathbb{E}[r(t)]$ | $r(0)e^{-\kappa t} + \theta(1-e^{-\kappa t})$ | Same |
| $\text{Var}[r(t)]$ | $\frac{\sigma^2}{2\kappa}[r(0)e^{-\kappa t}(1-e^{-\kappa t}) + \frac{\theta}{2}(1-e^{-\kappa t})^2]$ | $\frac{\sigma_{V}^2}{2\kappa}(1-e^{-2\kappa t})$ |
| $r(t)$ distribution | Scaled non-central $\chi^2$ | Normal |
| Stationary distribution | $\Gamma\!\left(\frac{2\kappa\theta}{\sigma^2}, \frac{\sigma^2}{2\kappa}\right)$ | $\mathcal{N}\!\left(\theta, \frac{\sigma_{V}^2}{2\kappa}\right)$ |
| Negative rates possible | **No** | Yes |

The stationary Gamma has shape $d/2 = 2\kappa\theta/\sigma^2$ and rate $2\kappa/\sigma^2$, giving mean $\theta$ and variance $\theta\sigma^2/(2\kappa)$ — note the extra factor of $\theta$ compared to Vasicek, making CIR less extreme in tail scenarios.

### 1.5 Exact Simulation via Non-Central Chi-Squared

The one-step conditional distribution has a closed-form expression:

$$r(t+\Delta t)\,\big|\,r(t) \;\sim\; c \cdot \chi^2\!\left(d,\; \lambda(r(t))\right)$$

$$c = \frac{\sigma^2(1 - e^{-\kappa\Delta t})}{4\kappa} \qquad d = \frac{4\kappa\theta}{\sigma^2} \qquad \lambda = \frac{r(t)\,e^{-\kappa\Delta t}}{c}$$

where $\sigma$ and $\theta$ are used in their **percent-convention units** throughout (see the note in `CIRProcess.__init__` about the $\sqrt{\%}$/year convention). $r(t+\Delta t) \geq 0$ structurally because $\chi^2(d,\lambda) \geq 0$.

The `CIRProcess.simulate()` production class samples this directly via `numpy.random.Generator.noncentral_chisquare`. **Antithetic variates are not supported** — the non-centrality parameter $\lambda$ depends on $r(t)$, so path pairs diverge immediately; increase $n_{\text{paths}}$ to compensate.


In [ ]:
# ── Visualise the key CIR properties ─────────────────────────────────────────

r_grid = np.linspace(0.0, 6.0, 300)
kappa, theta_ref, sigma_cir = 0.10, 2.5, 0.316   # σ_CIR ≈ σ_Vasicek/√θ = 0.5/√2.5
sigma_vasicek = sigma_cir * np.sqrt(theta_ref)    # equivalent Vasicek σ at r=θ

fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(13, 4))

# Left: diffusion coefficient σ√r vs Vasicek constant σ
ax = ax0
ax.plot(r_grid, sigma_cir * np.sqrt(r_grid), color='darkorange', lw=2.0,
        label=f'CIR: $\sigma\sqrt{{r}}$  (σ={sigma_cir:.3f})')
ax.axhline(sigma_vasicek, color='steelblue', lw=2.0, ls='--',
           label=f'Vasicek: $\sigma$  (σ={sigma_vasicek:.3f}%/√yr)')
ax.axvline(theta_ref, color='gray', lw=0.8, ls=':', alpha=0.7, label=f'θ={theta_ref}%')
ax.set_xlabel('r(t) (%)')
ax.set_ylabel('Diffusion coefficient (% / √year)')
ax.set_title('State-dependent vol: CIR vs Vasicek
(equal vol at r = θ)')
ax.legend(fontsize=7)
ax.set_xlim(0, 6)

# Middle: Feller condition boundaries
ax = ax1
sigma_grid_feller = np.linspace(0.05, 1.2, 200)
theta_feller_min  = sigma_grid_feller**2 / (2 * kappa)   # boundary 2κθ = σ²
ax.fill_between(sigma_grid_feller, theta_feller_min, 6.0, alpha=0.15, color='forestgreen',
                label='Feller satisfied: $2\kappa\theta > \sigma^2$')
ax.fill_between(sigma_grid_feller, 0, theta_feller_min, alpha=0.15, color='firebrick',
                label='Feller violated: zero accessible')
ax.plot(sigma_grid_feller, theta_feller_min, 'k-', lw=1.5, label='Boundary $2\kappa\theta = \sigma^2$')
ax.scatter([sigma_cir], [theta_ref], color='darkorange', s=80, zorder=5,
           label=f'Reference (σ={sigma_cir:.3f}, θ={theta_ref}%)')
ax.set_xlabel('σ (√%/√yr)')
ax.set_ylabel('θ (%)')
ax.set_title(f'Feller condition landscape  (κ={kappa})')
ax.legend(fontsize=7)
ax.set_xlim(0, 1.2); ax.set_ylim(0, 6)

# Right: stationary distributions — Gamma (CIR) vs Normal (Vasicek)
ax = ax2
r_plot = np.linspace(0, 6, 300)
# CIR stationary: Gamma(shape=2κθ/σ², rate=2κ/σ²) where σ is in percent-convention
# In percent units: shape = 2κθ/σ² = 2*0.1*2.5/0.316² = 5.0 (dimensionless ✓)
# scale = σ²/(2κ) = 0.316²/(0.2) = 0.499% (in percent)
shape_cir = 2*kappa*theta_ref / sigma_cir**2
scale_cir  = sigma_cir**2 / (2*kappa)
pdf_cir    = gamma_dist.pdf(r_plot, a=shape_cir, scale=scale_cir)

# Vasicek stationary: N(θ, σ_V²/(2κ))
std_vasicek = sigma_vasicek / np.sqrt(2*kappa)
pdf_vasicek = stats.norm.pdf(r_plot, theta_ref, std_vasicek)

ax.plot(r_plot, pdf_cir,    color='darkorange', lw=2.0, label=f'CIR: Γ(d/2={shape_cir:.1f}, scale={scale_cir:.2f}%)')
ax.plot(r_plot, pdf_vasicek,color='steelblue',  lw=2.0, ls='--', label=f'Vasicek: N({theta_ref}%, {std_vasicek:.2f}%)')
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('r  (%)')
ax.set_ylabel('Density')
ax.set_title('Stationary distributions: Gamma vs Normal
(both have mean θ=2.5%)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

d_feller = 4*kappa*theta_ref/sigma_cir**2
print(f"CIR parameters: κ={kappa}, θ={theta_ref}%, σ={sigma_cir:.3f}√%/√yr")
print(f"  d = 4κθ/σ² = {d_feller:.2f}  (degrees of freedom — must be > 2 for Feller)")
print(f"  Feller condition 2κθ > σ²: {2*kappa*theta_ref:.3f} > {sigma_cir**2:.3f}  "
      f"→ {'satisfied' if 2*kappa*theta_ref > sigma_cir**2 else 'VIOLATED'}")
print(f"  Stationary: mean={theta_ref}%,  std={np.sqrt(theta_ref*sigma_cir**2/(2*kappa)):.4f}%")
print(f"  Equivalent Vasicek σ at r=θ: {sigma_vasicek:.3f}%/√yr")


---
## 2. The Non-Central Chi-Squared Transition Density

### 2.1 From the SDE to the Distribution

The key result (Cox, Ingersoll, Ross 1985): the conditional distribution of $r(t+\Delta t)$ given $r(t)$ is a **scaled non-central chi-squared** distribution. This follows from writing the SDE solution in terms of squared Ornstein-Uhlenbeck processes.

The three parameters:
- **Scale** $c = \sigma^2(1-e^{-\kappa\Delta t})/(4\kappa)$ — in percent, shrinks as $\Delta t \to 0$
- **Degrees of freedom** $d = 4\kappa\theta/\sigma^2$ — constant, encodes the Feller condition
- **Non-centrality** $\lambda = r(t)\,e^{-\kappa\Delta t}/c$ — path-dependent, carries the current rate

**Consistency checks:**
$$\mathbb{E}[r(t+\Delta t)] = c(d+\lambda) = c\cdot d + r(t)\,e^{-\kappa\Delta t} = \underbrace{\theta(1-e^{-\kappa\Delta t})}_{\text{drift to } \theta} + \underbrace{r(t)\,e^{-\kappa\Delta t}}_{\text{decay of }r(t)} \checkmark$$

$$\mathbb{E}[r(t+\Delta t)\,\big|\,r(t)] = r(t)\,e^{-\kappa\Delta t} + \theta(1 - e^{-\kappa\Delta t})$$

This is **identical to the Vasicek conditional mean** — confirming that the mean dynamics are unchanged.

### 2.2 What Changes Compared to Vasicek

The **variance** of the transition is:
$$\text{Var}[r(t+\Delta t)\,\big|\,r(t)] = c^2(2(d+2\lambda)) = \frac{\sigma^2 r(t)}{\kappa}\,e^{-\kappa\Delta t}(1-e^{-\kappa\Delta t}) + \frac{\sigma^2\theta}{2\kappa}(1-e^{-\kappa\Delta t})^2$$

The variance **depends on $r(t)$** — at high rates, variance is higher. In Vasicek, the conditional variance is $\sigma^2(1-e^{-2\kappa\Delta t})/(2\kappa)$ — constant.

### 2.3 Antithetic Variates: Why They Are Not Available

In Vasicek, the stochastic increment is $v_{\Delta t} \cdot Z$. An antithetic twin uses $-Z$. This produces a path pair with opposite deviations that cancel in averages.

In CIR, the increment is $c \cdot \chi^2(d,\lambda(r(t)))$. The non-centrality $\lambda$ **depends on $r(t)$**, which diverges between twins after the first step. There is no simple symmetric pairing. The `CIRProcess` raises `NotImplementedError` if `antithetic=True` is requested — compensate with a larger $n_{\text{paths}}$.


In [ ]:
# ── Visualise the non-central chi-squared transition density ──────────────────
kappa_v, theta_v, sigma_v = 0.10, 2.5, 0.316
dt = 1.0   # 1-year horizon for clear visual separation

# Pre-compute transition parameters (all in percent convention)
c   = sigma_v**2 * (1 - np.exp(-kappa_v*dt)) / (4*kappa_v)   # scale in percent
d   = 4*kappa_v*theta_v / sigma_v**2                           # degrees of freedom

r_axis = np.linspace(0, 8, 500)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: transition densities for different starting rates r(0)
ax = ax0
for r0_val, color in [(0.5, 'steelblue'), (2.5, 'darkorange'), (5.0, 'firebrick')]:
    lam   = r0_val * np.exp(-kappa_v*dt) / c          # non-centrality (dimensionless)
    E_r1  = r0_val*np.exp(-kappa_v*dt) + theta_v*(1-np.exp(-kappa_v*dt))

    # Scaled non-central chi-squared PDF: if X ~ χ²(d,λ) then c·X has pdf f(x/c)/c
    pdf_vals = ncx2.pdf(r_axis / c, df=d, nc=lam) / c

    ax.plot(r_axis, pdf_vals, color=color, lw=1.8,
            label=f'r(0)={r0_val:.1f}%  (E[r(1Y)]={E_r1:.2f}%)')
ax.axvline(theta_v, color='black', lw=0.8, ls=':', alpha=0.5, label=f'θ={theta_v}%')
ax.set_xlabel('r(1Y) (%)')
ax.set_ylabel('Density')
ax.set_title(f'CIR transition density over Δt=1Y
'
             f'κ={kappa_v}, θ={theta_v}%, σ={sigma_v:.3f}√%/√yr  (d={d:.1f})')
ax.legend(fontsize=7)

# Right: compare CIR vs Vasicek transition densities at r(0)=2.5%
ax = ax1
r0_comp = 2.5
# CIR
lam_cir   = r0_comp * np.exp(-kappa_v*dt) / c
pdf_cir_t = ncx2.pdf(r_axis/c, df=d, nc=lam_cir) / c

# Vasicek (equivalent σ at r=θ)
sigma_vas = sigma_v * np.sqrt(theta_v)   # 0.5%/√year
exp_k     = np.exp(-kappa_v*dt)
mu_vas    = r0_comp*exp_k + theta_v*(1-exp_k)
var_vas   = sigma_vas**2 * (1-np.exp(-2*kappa_v*dt)) / (2*kappa_v)
pdf_vas   = stats.norm.pdf(r_axis, mu_vas, np.sqrt(var_vas))

ax.plot(r_axis, pdf_cir_t, color='darkorange', lw=2.0, label='CIR (non-central χ²)')
ax.plot(r_axis, pdf_vas,   color='steelblue',  lw=2.0, ls='--', label='Vasicek (Normal)')
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5, label='r = 0')
ax.set_xlabel('r(1Y) (%)')
ax.set_ylabel('Density')
ax.set_title(f'CIR vs Vasicek transition density from r(0)={r0_comp}%
'
             f'(same mean, different shape and tail at r=0)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"Transition parameters (κ={kappa_v}, σ={sigma_v:.3f}, dt=1Y):")
print(f"  Scale c         = σ²(1-e^{{-κΔt}})/(4κ) = {c:.5f}%")
print(f"  Degrees of freedom d = 4κθ/σ² = {d:.2f}")
for r0_val in [0.5, 2.5, 5.0]:
    lam   = r0_val * np.exp(-kappa_v*dt) / c
    E_r1  = r0_val*np.exp(-kappa_v*dt) + theta_v*(1-np.exp(-kappa_v*dt))
    print(f"  r(0)={r0_val:.1f}%  λ={lam:.2f}  E[r(1Y)]={E_r1:.3f}%")


---
## 3. CIR Affine Term Structure — Zero-Coupon Bond Pricing

### 3.1 The Bond Price Formula

CIR also has an affine term structure: $P(0,T) = A(T)\cdot\exp(-B(T)\cdot r_0)$.

The functions $B(T)$ and $A(T)$ differ from Vasicek because the bond pricing PDE now has a state-dependent diffusion term $\sigma^2 r$:

$$\frac{\partial P}{\partial t} + \kappa(\theta-r)\frac{\partial P}{\partial r} + \frac{\sigma^2 r}{2}\frac{\partial^2 P}{\partial r^2} - rP = 0$$

Substituting the affine ansatz $P = A\exp(-Br)$ and separating:

**$r$-coefficient** (Riccati ODE — note the $B^2$ term from $\sigma^2 r$):
$$B'(\tau) = 1/100 + \kappa B(\tau) - \frac{\sigma_{\text{dec}}^2}{2} B(\tau)^2, \quad B(0) = 0$$

The $B^2$ term (absent in Vasicek) makes this a **Bernoulli ODE**, solvable exactly with the substitution $B = 2u'/( \sigma_{\text{dec}}^2 u)$:

$$\boxed{B(T) = \frac{2(e^{\gamma T}-1)}{(\gamma+\kappa)(e^{\gamma T}-1)+2\gamma}}$$

where $\gamma = \sqrt{\kappa^2 + 2\sigma_{\text{dec}}^2}$.

**Constant coefficient** (linear ODE for $\ln A$):

$$\boxed{A(T) = \left[\frac{2\gamma\,e^{(\gamma+\kappa)T/2}}{(\gamma+\kappa)(e^{\gamma T}-1)+2\gamma}\right]^{2\kappa\theta/\sigma^2}}$$

where the exponent $2\kappa\theta/\sigma^2 = d/2$ uses $\theta$ and $\sigma$ in **percent-convention units** (the $100$ factors cancel).

### 3.2 Key Differences from Vasicek

<small>

| | Vasicek | CIR |
|---|---|---|
| $B(T)$ ODE | Linear: $B'=1/100-\kappa B$ | Bernoulli: $B'=1/100+\kappa B - \frac{\sigma_{\text{dec}}^2}{2}B^2$ |
| $B(T)$ limit as $T\to\infty$ | $1/(100\kappa)$ (year/%) | $2/(\gamma+\kappa)$ (year) |
| $A(T)$ form | Exponential of quadratic in $B$ | Power of rational function |
| $\gamma$ | $\kappa$ | $\sqrt{\kappa^2 + 2\sigma_{\text{dec}}^2} > \kappa$ |
| Long-run yield $R_\infty$ | $(\theta_{\text{dec}} - \sigma_{\text{dec}}^2/(2\kappa^2))\times 100$ | $2\kappa\theta/(\gamma+\kappa)$ |

</small>

The CIR long-run yield is always **positive** ($\theta > 0$, $\gamma > 0$, $\kappa > 0$), while Vasicek's $R_\infty$ can become negative for large $\sigma$.


In [ ]:
# ── CIR affine bond pricing ────────────────────────────────────────────────────
# Convention: θ and r0 in percent, σ in √%/√yr.
# Internally: σ_dec = σ/10 (not σ/100 — CIR has different units, see rates.py).
# Verification: should give P(0,T) ≈ exp(-r0/100 * T) for a flat curve at r0=θ.

def cir_gamma(kappa: float, sigma_pct: float) -> float:
    """γ = √(κ² + 2σ_dec²) where σ_dec = σ_pct/10 (CIR convention)."""
    sigma_d = sigma_pct / 10
    return np.sqrt(kappa**2 + 2 * sigma_d**2)

def cir_B(T: np.ndarray, kappa: float, sigma_pct: float) -> np.ndarray:
    """B(T) = 2(e^{γT}-1) / [(γ+κ)(e^{γT}-1) + 2γ]."""
    gamma = cir_gamma(kappa, sigma_pct)
    exp_gT = np.exp(gamma * T)
    denom  = (gamma + kappa) * (exp_gT - 1) + 2 * gamma
    return 2 * (exp_gT - 1) / denom

def cir_A(T: np.ndarray, kappa: float, theta_pct: float, sigma_pct: float) -> np.ndarray:
    """A(T) = [2γ exp((γ+κ)T/2) / denom]^{2κθ/σ²}.
    Exponent uses percent-convention: 2κθ_pct/σ_pct² (100s cancel)."""
    gamma  = cir_gamma(kappa, sigma_pct)
    h1     = (gamma + kappa) / 2
    exp_gT = np.exp(gamma * T)
    denom  = (gamma + kappa) * (exp_gT - 1) + 2 * gamma
    exponent = 2 * kappa * theta_pct / sigma_pct**2   # = d/2, dimensionless
    # Compute in log space to avoid overflow at long T
    log_base = np.log(2 * gamma) + h1 * T - np.log(denom)
    return np.exp(exponent * log_base)

def cir_zcb(T: np.ndarray, r0_pct: float,
             kappa: float, theta_pct: float, sigma_pct: float) -> np.ndarray:
    """P^CIR(0,T) = A(T) * exp(-B(T) * r0/100).  r0 in percent; /100 for discounting."""
    return cir_A(T, kappa, theta_pct, sigma_pct) * np.exp(-cir_B(T, kappa, sigma_pct) * r0_pct / 100)

def cir_yield(T: np.ndarray, r0_pct: float,
               kappa: float, theta_pct: float, sigma_pct: float) -> np.ndarray:
    """Continuously compounded zero yield R(0,T) in percent."""
    P = cir_zcb(T, r0_pct, kappa, theta_pct, sigma_pct)
    return -np.log(P) / T * 100

def cir_R_inf(kappa: float, theta_pct: float, sigma_pct: float) -> float:
    """Long-run zero yield R∞ = 2κθ/(γ+κ) in percent."""
    gamma = cir_gamma(kappa, sigma_pct)
    return 2 * kappa * theta_pct / (gamma + kappa)

# ── Sanity check: flat curve r0=θ should give R(T) ≈ r0 for all T ────────────
T_check = np.array([0.001, 1.0, 2.0, 5.0, 10.0, 20.0])
kappa_c, theta_c, sigma_c = 0.10, 2.5, 0.316

print("CIR flat-curve check: r0=θ=2.5%, expect R(T) close to 2.5%")
for T in T_check:
    R = cir_yield(np.array([T]), 2.5, kappa_c, theta_c, sigma_c)[0]
    print(f"  R({T:5.3f}Y) = {R:.4f}%")

R_inf = cir_R_inf(kappa_c, theta_c, sigma_c)
print(f"\nR∞ = 2κθ/(γ+κ) = {R_inf:.4f}%  (long-end yield, always positive)")
print(f"γ = {cir_gamma(kappa_c, sigma_c):.4f}  (vs κ={kappa_c} for Vasicek)")

In [ ]:
# ── CIR yield curves vs Vasicek: shape comparison ────────────────────────────
# Import the Vasicek functions from the companion notebook (or redefine briefly)

def vasicek_B_fn(T, kappa): return (1 - np.exp(-kappa*T)) / kappa
def vasicek_alpha_fn(T, kappa, theta_pct, sigma_pct):
    Bt=vasicek_B_fn(T,kappa); td,sd=theta_pct/100,sigma_pct/100
    Ri=td-sd**2/(2*kappa**2); return Ri*(Bt-T)-sd**2/(4*kappa)*Bt**2
def vasicek_yield_fn(T, r0, kappa, theta_pct, sigma_pct):
    P=np.exp(vasicek_alpha_fn(T,kappa,theta_pct,sigma_pct)-vasicek_B_fn(T,kappa)*r0/100)
    return -np.log(P)/T*100

T_grid = np.linspace(0.25, 30, 300)
kappa_c, theta_c, sigma_c = 0.10, 2.5, 0.316
sigma_v_equiv = sigma_c * np.sqrt(theta_c)   # same vol at r=θ

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: CIR vs Vasicek for different r0
ax = ax0
for r0_val, color in [(0.5, 'steelblue'), (2.5, 'darkorange'), (5.0, 'firebrick')]:
    R_cir = cir_yield(T_grid, r0_val, kappa_c, theta_c, sigma_c)
    R_vas = vasicek_yield_fn(T_grid, r0_val, kappa_c, theta_c, sigma_v_equiv)
    ax.plot(T_grid, R_cir, '-',  color=color, lw=1.8, label=f'CIR r(0)={r0_val}%')
    ax.plot(T_grid, R_vas, '--', color=color, lw=1.2, alpha=0.7)
ax.axhline(cir_R_inf(kappa_c, theta_c, sigma_c), color='black', lw=0.8, ls=':',
           alpha=0.6, label=f'CIR R∞={cir_R_inf(kappa_c,theta_c,sigma_c):.2f}%')
ax.plot([], [], 'k-',  lw=1.8, label='CIR (solid)')
ax.plot([], [], 'k--', lw=1.2, label='Vasicek (dashed)')
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Zero yield R(0,T) (%)')
ax.set_title('CIR vs Vasicek yield curves
(same κ, θ, equal vol at r=θ)')
ax.legend(fontsize=7)

# Right: effect of σ on CIR yield curve — long-end suppression vs Vasicek
ax = ax1
for sigma_val, color in [(0.20, 'steelblue'), (0.50, 'darkorange'), (0.90, 'firebrick')]:
    R_cir_s = cir_yield(T_grid, 2.5, kappa_c, theta_c, sigma_val)
    ax.plot(T_grid, R_cir_s, '-', color=color, lw=1.8,
            label=f'σ={sigma_val:.2f}√%/√yr  R∞={cir_R_inf(kappa_c,theta_c,sigma_val):.2f}%')

ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Zero yield R(0,T) (%)')
ax.set_title('Effect of σ on CIR yield curve
(κ=0.10, θ=2.5%, r(0)=2.5%)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print("Key insight: unlike Vasicek, R∞^CIR is always > 0 since 2κθ > 0 and γ+κ > 0.")
print("Vasicek can produce negative R∞ for large σ (convexity dominates).")
print(f"For σ=0.90: CIR R∞={cir_R_inf(kappa_c,theta_c,0.90):.2f}%  vs")
vRinf = (theta_c/100 - (sigma_v_equiv*0.90/sigma_c)**2/(100**2)/(2*kappa_c**2))*100
print(f"Vasicek R∞ would be: {vRinf:.2f}% (possibly negative at high σ)")


---
## 4. Calibration to the OIS Term Structure

### 4.1 The Same Least-Squares Approach as Vasicek

CIR has the same three free parameters ($\kappa$, $\theta$, $\sigma$) and the same 1-factor structure as Vasicek. The calibration approach is identical:

$$\min_{\kappa,\theta,\sigma} \sum_i w_i\bigl(R^{\text{CIR}}(T_i;\kappa,\theta,\sigma,r_0) - R_i^{\text{OIS}}\bigr)^2$$

with maturity-proportional weights $w_i = T_i/\sum_j T_j$ and $r_0$ anchored to the 1M→2M OIS forward rate.

### 4.2 The Feller Constraint

Unlike Vasicek, CIR has an explicit positivity-guaranteeing constraint. We enforce the Feller condition $2\kappa\theta > \sigma^2$ during optimisation to stay in the well-defined region. The `CIRProcess` issues a `UserWarning` at construction if violated.

### 4.3 The 1-Factor Limitation (Same as Vasicek)

Both CIR and Vasicek are constant-parameter, 1-factor models — they share the same structural inability to fit the full shape of the OIS curve. The calibration residuals at intermediate maturities will be similar. The **mean dynamics are identical** in both models; only the volatility structure differs. For OIS-exact fitting, Hull-White (QRE-44) is required.


In [ ]:
# ── Load OIS and calibrate CIR to zero yields ─────────────────────────────────
try:
    ois = OISCurve.from_processed(str(PROCESSED_DIR))
    print(ois.describe())
except FileNotFoundError:
    print("Processed OIS not found — using synthetic curve")
    data = pd.DataFrame(
        {"years":           [1/12, 2/12, 3/12, 6/12, 9/12, 1.0,  2.0,  3.0,  5.0,  10.0, 15.0],
         "zero_rate_pct":   [2.63, 2.61, 2.58, 2.48, 2.38, 2.30, 2.20, 2.15, 2.20,  2.40, 2.50],
         "discount_factor": [np.exp(-r/100*t) for r,t in zip(
                                [2.63,2.61,2.58,2.48,2.38,2.30,2.20,2.15,2.20,2.40,2.50],
                                [1/12,2/12,3/12,6/12,9/12,1,2,3,5,10,15])],
         "valuation_date":  ["2026-03-24"] * 11},
        index=["1M","2M","3M","6M","9M","12M","2Y","3Y","5Y","10Y","10Y+"])
    data.index.name = "maturity"
    ois = OISCurve(data)
    print(ois.describe())

r0 = ois.forward_rate(1/12, 2/12)
print(f"r(0) = {r0:.4f}%")

cal_maturities = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0])
ois_yields     = np.array([ois.zero_rate(T) for T in cal_maturities])
weights        = cal_maturities / cal_maturities.sum()

def calibration_objective(params: np.ndarray) -> float:
    kappa_c, theta_c, sigma_c = params
    if kappa_c <= 0 or sigma_c <= 0 or theta_c <= 0:
        return 1e10
    if 2 * kappa_c * theta_c <= sigma_c**2:
        # Penalise Feller violations smoothly rather than hard-blocking
        feller_penalty = 1e4 * (sigma_c**2 - 2*kappa_c*theta_c)
    else:
        feller_penalty = 0.0
    model_yields = cir_yield(cal_maturities, r0, kappa_c, theta_c, sigma_c)
    sse = np.sum(weights * (model_yields - ois_yields)**2) * 1e4
    return sse + feller_penalty

# Multi-start optimisation
theta_init = ois_yields[-1]
x0_list = [(0.05, theta_init, 0.25), (0.10, theta_init, 0.35), (0.20, theta_init, 0.50)]
best = None
for x0 in x0_list:
    res = minimize(calibration_objective, x0=np.array(x0), method='Nelder-Mead',
                   options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-10})
    if best is None or res.fun < best.fun:
        best = res

kappa_cal, theta_cal, sigma_cal = best.x
R_inf_cal = cir_R_inf(kappa_cal, theta_cal, sigma_cal)
feller_ok  = 2*kappa_cal*theta_cal > sigma_cal**2
d_cal      = 4*kappa_cal*theta_cal / sigma_cal**2

print(f"\nCalibrated CIR parameters:")
print(f"  κ   = {kappa_cal:.4f}  (half-life {np.log(2)/kappa_cal:.1f}y)")
print(f"  θ   = {theta_cal:.4f}%")
print(f"  σ   = {sigma_cal:.4f}√%/√yr")
print(f"  R∞  = 2κθ/(γ+κ) = {R_inf_cal:.4f}%")
print(f"  d   = 4κθ/σ² = {d_cal:.2f}  (degrees of freedom)")
print(f"  Feller 2κθ={2*kappa_cal*theta_cal:.4f} vs σ²={sigma_cal**2:.4f}  "
      f"→ {'satisfied' if feller_ok else 'VIOLATED'}")

model_yields_cal = cir_yield(cal_maturities, r0, kappa_cal, theta_cal, sigma_cal)
residuals_bps    = (model_yields_cal - ois_yields) * 100
rmse_bps         = np.sqrt(np.mean(residuals_bps**2))
print(f"\nCalibration residuals (model − OIS, bps):")
print(f"  {'Maturity':>8}  {'OIS':>8}  {'CIR':>8}  {'Error':>8}")
for T, Ro, Rm, err in zip(cal_maturities, ois_yields, model_yields_cal, residuals_bps):
    print(f"  {T:>8.2f}Y  {Ro:>8.4f}%  {Rm:>8.4f}%  {err:>+8.2f}bps")
print(f"  RMSE = {rmse_bps:.2f} bps")


In [ ]:
# ── Calibration quality plot ──────────────────────────────────────────────────
T_dense = np.linspace(0.1, 30, 500)
R_cir_dense = cir_yield(T_dense, r0, kappa_cal, theta_cal, sigma_cal)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.5))

ax = ax0
ax.plot(T_dense, R_cir_dense, '-', color='darkorange', lw=2.0, label='CIR (calibrated)')
ax.plot(cal_maturities, ois_yields, 'o', color='black', ms=6, zorder=5, label='OIS pillars')
ax.axhline(R_inf_cal, lw=0.8, ls=':', color='darkorange', alpha=0.6,
           label=f'R∞={R_inf_cal:.2f}%')
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Zero yield (%)')
ax.set_title('CIR calibration to OIS zero curve')
ax.legend()

ax = ax1
colors = ['firebrick' if abs(e) > 5 else 'darkorange' for e in residuals_bps]
ax.bar(cal_maturities, residuals_bps, width=0.6, color=colors, alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(5,  color='firebrick', lw=0.8, ls='--', alpha=0.5)
ax.axhline(-5, color='firebrick', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Residual (bps)')
ax.set_title(f'Calibration residuals — RMSE = {rmse_bps:.2f} bps\n'
             f'(±5 bps threshold shown)')
plt.tight_layout()
plt.show()

print("Observation: like Vasicek, CIR cannot fit arbitrary curvature (1-factor model).")
print("Residual pattern is similar — the belly error comes from the same structural limitation.")


---
## 5. Path Simulation with Calibrated Parameters

### 5.1 What to Verify

With calibrated $(\hat{\kappa}, \hat{\theta}, \hat{\sigma})$, the simulation should satisfy:
1. **Non-negativity** — all simulated rates $\geq 0$, verified empirically
2. **Mean convergence** — $\bar{r}(t) \to \mathbb{E}[r(t)]$ as $n_{\text{paths}}$ grows
3. **Stationary Gamma** — the terminal distribution should fit a $\Gamma(d/2, \sigma^2/(2\kappa))$ distribution

Note: because `CIRProcess` does not support antithetic variates, we use a larger path count ($n_{\text{paths}} = 5{,}000$) to achieve comparable standard error to a 2,000-path antithetic simulation.


In [ ]:
# ── Simulate CIR paths with calibrated parameters ─────────────────────────────
import warnings

cir_proc = CIRProcess(kappa=kappa_cal, theta=theta_cal, sigma=sigma_cal)
print(cir_proc.describe())

T_sim   = 10.0
n_steps = 120    # monthly — same as HW for comparability
n_paths = 5000   # larger n since no antithetic support
t_grid  = np.linspace(0, T_sim, n_steps + 1)

paths = cir_proc.simulate(
    x0        = r0,
    T         = T_sim,
    n_steps   = n_steps,
    n_paths   = n_paths,
    antithetic= False,   # CIR exact simulation does not support antithetic
    seed      = RNG_SEED,
)

# ── Verify non-negativity ─────────────────────────────────────────────────────
neg_count = (paths < 0).sum()
print(f"\nNon-negativity check: {neg_count} negative values out of "
      f"{paths.size:,}  ({neg_count/paths.size:.2e})")

# ── Theoretical mean ──────────────────────────────────────────────────────────
mu_t = r0 * np.exp(-kappa_cal * t_grid) + theta_cal*(1 - np.exp(-kappa_cal * t_grid))

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: fan chart
ax = ax0
for i in range(40):
    ax.plot(t_grid, paths[i], lw=0.3, alpha=0.2, color='darkorange')
q5,  q95 = np.percentile(paths, [5, 95], axis=0)
q25, q75 = np.percentile(paths, [25, 75], axis=0)
ax.fill_between(t_grid, q5, q95, color='darkorange', alpha=0.10, label='5–95th pctile')
ax.fill_between(t_grid, q25, q75, color='darkorange', alpha=0.20, label='25–75th pctile')
ax.plot(t_grid, paths.mean(axis=0), '-',  color='black', lw=1.5, label='Simulated mean')
ax.plot(t_grid, mu_t,               '--', color='steelblue', lw=1.5,
        label='Theoretical E[r(t)]')
ax.axhline(theta_cal, lw=0.8, ls=':', color='gray', alpha=0.7, label=f'θ={theta_cal:.2f}%')
ax.axhline(0, lw=0.8, color='black', alpha=0.3)
ax.set_xlabel('t (years)')
ax.set_ylabel('r(t) (%)')
ax.set_title(f'CIR fan chart — {n_paths:,} paths, no antithetic\n'
             f'κ={kappa_cal:.3f}, θ={theta_cal:.3f}%, σ={sigma_cal:.3f}√%/√yr')
ax.legend(fontsize=7)

# Right: terminal distribution vs Gamma stationary
ax = ax1
terminal = paths[:, -1]
r_range  = np.linspace(0, terminal.max()*1.1, 300)

# Stationary Gamma shape and scale in percent
shape_stat = 2*kappa_cal*theta_cal / sigma_cal**2   # = d/2
scale_stat = sigma_cal**2 / (2*kappa_cal)
pdf_gamma  = gamma_dist.pdf(r_range, a=shape_stat, scale=scale_stat)

ax.hist(terminal, bins=60, density=True, color='darkorange', alpha=0.6,
        label=f'Simulated r({T_sim:.0f}Y)')
ax.plot(r_range, pdf_gamma, '-', color='black', lw=2.0,
        label=f'Stationary Γ(d/2={shape_stat:.1f}, scale={scale_stat:.3f}%)')
ax.set_xlabel(f'r({T_sim:.0f}Y) (%)')
ax.set_ylabel('Density')
ax.set_title(f'Terminal distribution vs stationary Gamma\n(T={T_sim:.0f}Y)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"Terminal statistics (T={T_sim}Y, {n_paths:,} paths):")
print(f"  Simulated:   mean={terminal.mean():.4f}%  std={terminal.std():.4f}%")
print(f"  E[r({T_sim}Y)]:  {mu_t[-1]:.4f}%  (theoretical)")
print(f"  Stationary:  mean={theta_cal:.4f}%  std={np.sqrt(theta_cal*sigma_cal**2/(2*kappa_cal)):.4f}%")
print(f"  Negative rates: {(terminal < 0).sum()}")


---
## 6. CIR vs Vasicek — Structural Comparison

### 6.1 Mean Paths: Identical

Both models share the same conditional mean:
$$\mathbb{E}[r(t)\,\big|\,r(0)] = r(0)\,e^{-\kappa t} + \theta(1-e^{-\kappa t})$$

If you only care about **expected rates** (e.g., for discounting a bond in a deterministic scenario), CIR and Vasicek are indistinguishable given the same $\kappa$ and $\theta$.

### 6.2 Variance Structure: Different

<small>


| | CIR | Vasicek |
|---|---|---|
| Conditional variance | Depends on $r(t)$ — increases with rate level | Constant — independent of $r(t)$ |
| Stationary variance | $\theta\sigma^2/(2\kappa)$ | $\sigma_V^2/(2\kappa)$ |
| Implication | Higher rates → more uncertainty; near-zero → nearly deterministic | Same uncertainty at all rate levels |

</small>


### 6.3 When CIR Is Preferable

- **Regulatory stress tests** where rates are constrained to be non-negative (BCBS, EBA)
- **Emerging market contexts** where nominal rates are well above zero
- **Historical period calibration** (e.g., 1980s high-rate US/EUR environments)
- **Academic reference** for fixed income modelling in textbooks

### 6.4 When Vasicek or HW Is Preferable

- **EUR negative rate environments** (2014–2022): CIR rates are bounded at zero which conflicts with market reality
- **XVA production systems** where exact OIS fitting is required (→ use HW)
- **Swaption calibration** (→ HW has the Jamshidian formula; CIR swaption pricing is more complex)
- **Antithetic variance reduction** is needed for large exposure calculations


In [ ]:
# ── CIR vs Vasicek side-by-side comparison ────────────────────────────────────
# Use same (κ, θ) with σ chosen so both have equal vol at r=θ.
# Then compare: fan charts, terminal distributions, ZCB prices.

kappa_cmp, theta_cmp  = kappa_cal, theta_cal
sigma_cir_cmp  = sigma_cal
sigma_vas_cmp  = sigma_cir_cmp * np.sqrt(theta_cmp)   # equal vol at r=θ

print(f"Comparison parameters (equal vol at r=θ={theta_cmp:.2f}%):")
print(f"  CIR: κ={kappa_cmp:.3f}, θ={theta_cmp:.3f}%, σ={sigma_cir_cmp:.4f}√%/√yr")
print(f"  Vasicek: κ={kappa_cmp:.3f}, θ={theta_cmp:.3f}%, σ={sigma_vas_cmp:.4f}%/√yr")

from quant_risk.models.rates import VasicekProcess

vas_proc = VasicekProcess(kappa=kappa_cmp, theta=theta_cmp, sigma=sigma_vas_cmp)
cir_proc2 = CIRProcess(kappa=kappa_cmp, theta=theta_cmp, sigma=sigma_cir_cmp)

T_cmp, n_st, n_pa = 5.0, 60, 2000
paths_vas = vas_proc.simulate(x0=r0, T=T_cmp, n_steps=n_st, n_paths=n_pa,
                               antithetic=True, seed=RNG_SEED)
paths_cir = cir_proc2.simulate(x0=r0, T=T_cmp, n_steps=n_st, n_paths=n_pa,
                                antithetic=False, seed=RNG_SEED)

t_cmp = np.linspace(0, T_cmp, n_st + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Fan charts
for ax, paths, label, color in zip(
        axes[:2], [paths_vas, paths_cir],
        ['Vasicek', 'CIR'], ['steelblue', 'darkorange']):
    for i in range(25):
        ax.plot(t_cmp, paths[i], lw=0.3, alpha=0.25, color=color)
    q5,q95  = np.percentile(paths, [5,95], axis=0)
    q25,q75 = np.percentile(paths, [25,75], axis=0)
    ax.fill_between(t_cmp, q5, q95, color=color, alpha=0.10)
    ax.fill_between(t_cmp, q25, q75, color=color, alpha=0.22)
    ax.plot(t_cmp, paths.mean(axis=0), lw=1.5, color='black')
    ax.axhline(0, lw=0.8, color='black', alpha=0.3, ls='--')
    ax.axhline(theta_cmp, lw=0.8, ls=':', color='gray', alpha=0.7)
    ax.set_xlabel('t (years)')
    ax.set_ylabel('r(t) (%)')
    ax.set_title(f'{label} paths
(κ={kappa_cmp:.2f}, θ={theta_cmp:.2f}%, {n_pa} paths)')

# Terminal distribution comparison
ax = ax2
ax.hist(paths_vas[:,-1], bins=50, density=True, alpha=0.6, color='steelblue',
        label='Vasicek (Normal)')
ax.hist(paths_cir[:,-1], bins=50, density=True, alpha=0.6, color='darkorange',
        label='CIR (Gamma)')
ax.axvline(0, color='black', lw=1.0, ls='--', alpha=0.5)
ax.set_xlabel(f'r({T_cmp:.0f}Y) (%)')
ax.set_ylabel('Density')
ax.set_title(f'Terminal distribution comparison (T={T_cmp:.0f}Y)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

neg_vas = (paths_vas[:,-1] < 0).sum()
neg_cir = (paths_cir[:,-1] < 0).sum()
print(f"\nNegative terminal rates: Vasicek={neg_vas}/{n_pa}  CIR={neg_cir}/{n_pa}")


In [ ]:
# ── Production class: CIRProcess ─────────────────────────────────────────────
# CIRProcess is in src/quant_risk/models/rates.py (QRE-46)
# MCSimulator wraps it for ZCB pricing and exposure profiles

from quant_risk.models.simulator import MCSimulator
import warnings

# Suppress the UserWarning check during construction — we already verified Feller
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    sim = MCSimulator(
        process   = CIRProcess(kappa=kappa_cal, theta=theta_cal, sigma=sigma_cal),
        x0        = r0,
        T         = 10.0,
        n_steps   = 120,
        n_paths   = 5000,    # larger to compensate for no antithetic
        antithetic= False,   # must be False for CIRProcess
        seed      = RNG_SEED,
    )
print(sim.describe())

# ZCB prices via MCSimulator.sdf() vs CIR analytical formula
print("\nZCB prices: MCSimulator vs CIR analytical vs OIS")
print(f"  {'T':>5}  {'MC':>10}  {'CIR formula':>12}  {'OIS':>10}  {'MC-OIS (bps)':>14}")
for T_p in [1.0, 2.0, 5.0, 10.0]:
    P_mc    = sim.sdf(T_p).mean()
    P_cir_f = cir_zcb(np.array([T_p]), r0, kappa_cal, theta_cal, sigma_cal)[0]
    P_ois   = ois.discount(T_p)
    print(f"  {T_p:>5.1f}Y  {P_mc:>10.6f}  {P_cir_f:>12.6f}  {P_ois:>10.6f}  "
          f"{(P_mc-P_ois)*1e4:>+14.2f}")

print("\nNote: CIR is not calibrated to reproduce OIS DFs exactly (unlike HW).")
print("The MC-OIS difference is the same as the calibration yield residual.")


---
## Summary
<small>

| Step | Key formula | Notes |
|---|---|---|
| **SDE** | $dr = \kappa(\theta-r)dt + \sigma\sqrt{r}\,dW$ | σ in $\sqrt{\%}$/year (different from Vasicek) |
| **Feller** | $2\kappa\theta > \sigma^2$ ↔ $d = 4\kappa\theta/\sigma^2 > 2$ | Guarantees $r(t) > 0$ a.s. |
| **Exact simulation** | $r(t+\Delta t)\,\big|\,r(t) \sim c\cdot\chi^2(d,\lambda)$ | $c$, $d$ constant; $\lambda = r(t)e^{-\kappa\Delta t}/c$ |
| **Stationary** | $r(\infty) \sim \Gamma(d/2,\, \sigma^2/(2\kappa))$ | Non-Gaussian; $\Pr(r<0) = 0$ |
| **Bond price** | $P(0,T) = A(T)\exp(-B(T)\cdot r_0/100)$ | $\gamma = \sqrt{\kappa^2+2\sigma_{\text{dec}}^2}$ enters $A$, $B$ |
| **Long-run yield** | $R_\infty = 2\kappa\theta/(\gamma+\kappa)$ | Always positive; compare to Vasicek's $R_\infty = \theta - \sigma_{\text{dec}}^2/(2\kappa^2)$ |
| **Calibration** | OIS least-squares with Feller penalty | Same 1-factor limitation as Vasicek |
| **Production** | `CIRProcess(κ, θ, σ)` in `src/quant_risk/models/rates.py` | No antithetic; use $n_{\text{paths}} \geq 5{,}000$ |

</small>


**The CIR model in one sentence:** the same mean dynamics as Vasicek, but the variance collapses near zero (preventing negative rates), the stationary distribution is Gamma, and exact simulation uses the non-central chi-squared rather than the Gaussian distribution.

---

**With QRE-43 (Vasicek), QRE-44 (HW), and QRE-45 (CIR) complete**, the three short rate models for Module 3 are documented. The production classes `VasicekProcess`, `HullWhiteProcess`, and `CIRProcess` from QRE-46 back all three notebooks. The `MCSimulator` from QRE-51 provides a unified path-simulation and exposure-profile engine for all of them, feeding directly into the XVA notebooks (QRE-52–57).
